In [0]:
%run /Workspace/Users/kaustuvdev8123@outlook.com/utilClass/Encryption


In [0]:
base_path = "abfss://data@stdevwesteuropewrmp.dfs.core.windows.net"

raw_path = base_path + "/raw_data/hotel-weather"

bronze_write_path = base_path + "/bronze/hotel_weather_raw"

schema_location_path = base_path + "/metadata/bronze/schema"

hotel_weather_df = spark.readStream\
    .format("cloudfiles")\
    .option('cloudFiles.format', 'parquet')\
    .option('cloudFiles.schemaLocation', schema_location_path)\
    .option('inferSchema', True)\
    .load(raw_path)

columns_to_be_encrypted = ("address", "city", "country", "name")

encrypted_hotel_weather = Encrypter().encrypt_columns(
    hotel_weather_df, *columns_to_be_encrypted
)

encrypted_hotel_weather.writeStream\
    .format("delta")\
    .option("checkpointLocation", base_path + "/checkpoint/bronze_checkpoint")\
    .start(bronze_write_path)

